# LeetCode 2012 — Sum of Beauty in the Array bằng Python

Link đề: https://leetcode.com/problems/sum-of-beauty-in-the-array/ · Độ khó: **Medium** · Chủ đề: Array, Prefix Sum (prefix max / suffix min)

Skeleton LeetCode:

```python
class Solution:
    def sumOfBeauties(self, nums: List[int]) -> int:
        ...
```

## 1. Mục tiêu học

1. Dịch điều kiện "lớn hơn **mọi** phần tử bên trái và nhỏ hơn **mọi** phần tử bên phải" thành `max(trái) < x < min(phải)`.
2. Xây dựng **mảng max tiền tố** và **mảng min hậu tố** trong `O(n)` (kể cả bằng `itertools.accumulate`).
3. Tối ưu bộ nhớ: chỉ giữ **một** mảng hậu tố, còn max tiền tố cập nhật "trên đường đi".
4. Cẩn thận với **dấu bất đẳng thức nghiêm ngặt**, phạm vi chỉ số `1..n−2`, và quan hệ **if / elif** giữa hai mức điểm.
5. Kiểm thử đối chiếu brute force `O(n²)` với dữ liệu ngẫu nhiên có nhiều giá trị lặp.

## 2. Đề bài & ví dụ

Cho mảng `nums` (đánh số từ 0). Với mỗi chỉ số `i` trong `1 ≤ i ≤ n − 2`, **độ đẹp** của `nums[i]` là:

- `2` nếu `nums[j] < nums[i] < nums[k]` với **mọi** `j < i` và **mọi** `k > i`;
- `1` nếu `nums[i−1] < nums[i] < nums[i+1]` và điều kiện trên **không** thỏa;
- `0` nếu không thỏa điều kiện nào.

Trả về **tổng độ đẹp** của mọi `nums[i]` với `1 ≤ i ≤ n − 2`.

**Ví dụ 1:** `[1,2,3]` → `2` (độ đẹp của `nums[1] = 2` là 2).

**Ví dụ 2:** `[2,4,6,4]` → `1` (`nums[1] = 4`: `2 < 4 < 6` nhưng `4` không nhỏ hơn `nums[3] = 4` ⇒ 1; `nums[2] = 6`: `6 < 4` sai ⇒ 0).

**Ví dụ 3:** `[3,2,1]` → `0`.

**Ràng buộc:** `3 ≤ n ≤ 10⁵`, `1 ≤ nums[i] ≤ 10⁵`.

In [1]:
import random
import time
from itertools import accumulate
from typing import List

examples = [([1, 2, 3], 2), ([2, 4, 6, 4], 1), ([3, 2, 1], 0)]
for nums, exp in examples:
    print(nums, "-> kỳ vọng", exp)

[1, 2, 3] -> kỳ vọng 2
[2, 4, 6, 4] -> kỳ vọng 1
[3, 2, 1] -> kỳ vọng 0


## 3. Kiến thức nền

### 3.1 "Lớn hơn mọi phần tử" ⇔ "lớn hơn phần tử lớn nhất"

`x > a_j` với mọi `j` ⇔ `x > max(a)`. Tương tự `x < a_k` với mọi `k` ⇔ `x < min(a)`. Một phép so sánh thay cho cả vòng lặp — nếu ta có sẵn `max`/`min`.

In [2]:
left, x, right = [1, 3, 2], 4, [7, 5, 9]
print(all(v < x for v in left), x > max(left))
print(all(x < v for v in right), x < min(right))

True True
True True


### 3.2 Mảng max tiền tố và min hậu tố

- `pre[i] = max(nums[0..i])`: `pre[i] = max(pre[i−1], nums[i])`.
- `suf[i] = min(nums[i..n−1])`: `suf[i] = min(suf[i+1], nums[i])`, tính từ phải sang trái.

Khi đó "mọi phần tử bên trái `i`" là `pre[i−1]`, "mọi phần tử bên phải `i`" là `suf[i+1]`.

In [3]:
nums = [2, 4, 6, 4]
n = len(nums)
pre = nums[:]
for i in range(1, n):
    pre[i] = max(pre[i - 1], nums[i])
suf = nums[:]
for i in range(n - 2, -1, -1):
    suf[i] = min(suf[i + 1], nums[i])
print("nums:", nums, "| pre (max tiền tố):", pre, "| suf (min hậu tố):", suf)

nums: [2, 4, 6, 4] | pre (max tiền tố): [2, 4, 6, 6] | suf (min hậu tố): [2, 4, 4, 4]


### 3.3 `itertools.accumulate` với hàm `max` / `min`

`accumulate(a, max)` cho dãy max tiền tố. Với min hậu tố: đảo mảng, `accumulate(..., min)`, rồi đảo lại.

In [4]:
nums = [2, 4, 6, 4]
print(list(accumulate(nums, max)))
print(list(accumulate(reversed(nums), min))[::-1])

[2, 4, 6, 6]
[2, 4, 4, 4]


### 3.4 `if / elif` — mức điểm loại trừ nhau

Đề nói độ đẹp `1` chỉ khi điều kiện `2` **không** thỏa. Vì điều kiện 2 kéo theo điều kiện 1 (nếu lớn hơn mọi bên trái thì lớn hơn `nums[i−1]`...), nên ta viết `if cond2: +2 elif cond1: +1` — tuyệt đối không cộng cả hai.

In [5]:
def score(cond2, cond1):
    if cond2:
        return 2
    elif cond1:
        return 1
    return 0
print(score(True, True), score(False, True), score(False, False))

2 1 0


## 4. Ý tưởng nền tảng / trực giác

Tính tay `nums = [1, 5, 2, 3, 4, 6]`:

| i | nums[i] | max trái | min phải | điều kiện 2? | nums[i−1] < nums[i] < nums[i+1]? | độ đẹp |
|---|---|---|---|---|---|---|
| 1 | 5 | 1 | 2 | 1 < 5 < 2? **không** | 1 < 5 < 2? không | 0 |
| 2 | 2 | 5 | 3 | 5 < 2? không | 5 < 2? không | 0 |
| 3 | 3 | 5 | 4 | không | 2 < 3 < 4 ✓ | 1 |
| 4 | 4 | 5 | 6 | không | 3 < 4 < 6 ✓ | 1 |

Tổng `2`. Nhận xét: số `5` "chặn" mọi chỉ số phía sau khỏi điểm 2. Điểm 2 chỉ phụ thuộc vào **max của đoạn trái** và **min của đoạn phải** — hai đại lượng tính trước được.

In [6]:
nums = [1, 5, 2, 3, 4, 6]
for i in range(1, len(nums) - 1):
    L, R = max(nums[:i]), min(nums[i + 1:])
    print(f"i={i}: nums[i]={nums[i]}, max trái={L}, min phải={R}, "
          f"đk2={L < nums[i] < R}, đk1={nums[i - 1] < nums[i] < nums[i + 1]}")

i=1: nums[i]=5, max trái=1, min phải=2, đk2=False, đk1=False
i=2: nums[i]=2, max trái=5, min phải=3, đk2=False, đk1=False
i=3: nums[i]=3, max trái=5, min phải=4, đk2=False, đk1=True
i=4: nums[i]=4, max trái=5, min phải=6, đk2=False, đk1=True


## 5. Cách 1 — Brute Force / baseline: tính lại max/min cho từng i

**Ý tưởng:** làm đúng định nghĩa: với mỗi `i`, duyệt toàn bộ bên trái và bên phải.

In [7]:
def beauty_brute(nums):
    n = len(nums)
    total = 0
    for i in range(1, n - 1):
        if all(nums[j] < nums[i] for j in range(i)) and all(nums[i] < nums[k] for k in range(i + 1, n)):
            total += 2
        elif nums[i - 1] < nums[i] < nums[i + 1]:
            total += 1
    return total

for nums, exp in examples:
    got = beauty_brute(nums); print(nums, "->", got); assert got == exp

[1, 2, 3] -> 2
[2, 4, 6, 4] -> 1
[3, 2, 1] -> 0


**Độ phức tạp:** mỗi `i` tốn `O(n)` ⇒ `O(n²)`; với `n = 10⁵` khoảng `10¹⁰` phép so sánh (worst case khi mảng tăng dần, `all` không dừng sớm) ⇒ TLE. Space `O(1)`.

**Nút thắt:** `max(nums[0..i−1])` của `i` và của `i+1` chỉ khác nhau đúng một phần tử — ta đang tính lại từ đầu những thứ đã biết.

In [8]:
for n in (1000, 2000, 4000):
    arr = list(range(1, n + 1))          # tăng dần: worst case cho brute force
    t = time.perf_counter(); beauty_brute(arr)
    print(f"n={n}: {time.perf_counter() - t:.3f}s")

n=1000: 0.030s
n=2000: 0.121s


n=4000: 0.478s


## 6. Ý tưởng thuật toán / bước đột phá

**Insight:** điều kiện 2 ⇔ `pre[i−1] < nums[i] < suf[i+1]`. Hai mảng `pre`, `suf` tính trong `O(n)` bằng truy hồi một bước.

**Quy tắc "nếu… thì…":**
- **Nếu** `pre[i−1] < nums[i] < suf[i+1]` **thì** cộng `2`.
- **Ngược lại, nếu** `nums[i−1] < nums[i] < nums[i+1]` **thì** cộng `1`.
- **Ngược lại** cộng `0`.

**Invariant khi duyệt trái → phải (cách tiết kiệm bộ nhớ):** đầu vòng lặp tại `i`, biến `left_max` = `max(nums[0..i−1])`. Sau khi xử lý `i`, cập nhật `left_max = max(left_max, nums[i])`.

**Quan sát thêm:** điều kiện 2 đúng ⇒ điều kiện 1 đúng (vì `nums[i−1] ≤ pre[i−1]` và `nums[i+1] ≥ suf[i+1]`), nên các mức điểm "lồng nhau".

## 7. Các cách tiếp theo

### 7.1 Cách 2 — Hai mảng `pre` và `suf`

**Giải thích từng đoạn:**
1. `pre[i]` = max tiền tố, `suf[i]` = min hậu tố.
2. Duyệt `i = 1..n−2` và áp dụng quy tắc ở mục 6.

In [9]:
def beauty_two_arrays(nums):
    n = len(nums)
    pre = [0] * n; suf = [0] * n
    pre[0] = nums[0]
    for i in range(1, n):
        pre[i] = max(pre[i - 1], nums[i])
    suf[n - 1] = nums[n - 1]
    for i in range(n - 2, -1, -1):
        suf[i] = min(suf[i + 1], nums[i])
    total = 0
    for i in range(1, n - 1):
        if pre[i - 1] < nums[i] < suf[i + 1]:
            total += 2
        elif nums[i - 1] < nums[i] < nums[i + 1]:
            total += 1
    return total

for nums, exp in examples:
    assert beauty_two_arrays(nums) == exp
print([beauty_two_arrays(nums) for nums, _ in examples], "| ví dụ tay:", beauty_two_arrays([1, 5, 2, 3, 4, 6]))

[2, 1, 0] | ví dụ tay: 2


In [10]:
class SolutionTwoArrays:
    def sumOfBeauties(self, nums: List[int]) -> int:
        n = len(nums)
        pre = list(accumulate(nums, max))
        suf = list(accumulate(reversed(nums), min))[::-1]
        total = 0
        for i in range(1, n - 1):
            if pre[i - 1] < nums[i] < suf[i + 1]:
                total += 2
            elif nums[i - 1] < nums[i] < nums[i + 1]:
                total += 1
        return total

print([SolutionTwoArrays().sumOfBeauties(nums) for nums, _ in examples])

[2, 1, 0]


**Độ phức tạp Cách 2:** ba lượt duyệt `O(n)` ⇒ Time `O(n)`, Space `O(n)` (hai mảng).

### 7.2 Cách 3 — Một mảng `suf`, max tiền tố "chạy theo"

Ta cần `pre[i−1]` đúng lúc đang đứng ở `i` khi duyệt trái → phải, nên chỉ cần một biến `left_max`. Mảng `suf` vẫn phải tính trước (vì cần thông tin "tương lai").

In [11]:
def beauty_one_array(nums):
    n = len(nums)
    suf = [0] * (n + 1)
    suf[n] = float("inf")
    for i in range(n - 1, -1, -1):
        suf[i] = min(suf[i + 1], nums[i])
    total, left_max = 0, nums[0]
    for i in range(1, n - 1):
        if left_max < nums[i] < suf[i + 1]:
            total += 2
        elif nums[i - 1] < nums[i] < nums[i + 1]:
            total += 1
        left_max = max(left_max, nums[i])
    return total

for nums, exp in examples:
    assert beauty_one_array(nums) == exp
print([beauty_one_array(nums) for nums, _ in examples])

[2, 1, 0]


In [12]:
class SolutionOneArray:
    def sumOfBeauties(self, nums: List[int]) -> int:
        n = len(nums)
        suf = nums[:]
        for i in range(n - 2, -1, -1):
            suf[i] = min(suf[i + 1], nums[i])
        total, left_max = 0, nums[0]
        for i in range(1, n - 1):
            x = nums[i]
            if left_max < x < suf[i + 1]:
                total += 2
            elif nums[i - 1] < x < nums[i + 1]:
                total += 1
            if x > left_max:
                left_max = x
        return total

print([SolutionOneArray().sumOfBeauties(nums) for nums, _ in examples])

[2, 1, 0]


**Độ phức tạp Cách 3:** Time `O(n)`, Space `O(n)` cho một mảng (giảm một nửa so với Cách 2). Không thể xuống `O(1)` bộ nhớ một cách đơn giản vì `min` hậu tố cần nhìn trước.

### 7.3 Cách 4 — Đánh dấu "điểm 2" bằng hai mảng boolean, rồi cộng

Một cách viết khác tách bạch hai câu hỏi: `good_left[i]` = `nums[i]` lớn hơn mọi phần tử trước nó (**kỷ lục mới** từ trái); `good_right[i]` = `nums[i]` nhỏ hơn mọi phần tử sau nó (kỷ lục từ phải). Điểm 2 ⇔ cả hai đúng. Cách này dễ mở rộng khi đề hỏi "đếm số phần tử là kỷ lục từ cả hai phía".

In [13]:
def beauty_flags(nums):
    n = len(nums)
    good_left = [False] * n; best = float("-inf")
    for i, x in enumerate(nums):
        good_left[i] = x > best; best = max(best, x)
    good_right = [False] * n; best = float("inf")
    for i in range(n - 1, -1, -1):
        good_right[i] = nums[i] < best; best = min(best, nums[i])
    total = 0
    for i in range(1, n - 1):
        if good_left[i] and good_right[i]:
            total += 2
        elif nums[i - 1] < nums[i] < nums[i + 1]:
            total += 1
    return total

for nums, exp in examples:
    assert beauty_flags(nums) == exp
print([beauty_flags(nums) for nums, _ in examples], "| ví dụ tay:", beauty_flags([1, 5, 2, 3, 4, 6]))

[2, 1, 0] | ví dụ tay: 2


**Độ phức tạp Cách 4:** Time `O(n)`, Space `O(n)`.

## 8. TRACE từng bước

Trace Cách 3 trên `nums = [1, 5, 2, 3, 4, 6]`. Trước hết `suf = [1, 2, 2, 3, 4, 6]` (min hậu tố).

| i | x | left_max | suf[i+1] | đk2 | đk1 (láng giềng) | cộng | total | left_max mới |
|---|---|---|---|---|---|---|---|---|
| 1 | 5 | 1 | 2 | 1<5<2 ✗ | 1<5<2 ✗ | 0 | 0 | 5 |
| 2 | 2 | 5 | 3 | ✗ | 5<2 ✗ | 0 | 0 | 5 |
| 3 | 3 | 5 | 4 | ✗ | 2<3<4 ✓ | 1 | 1 | 5 |
| 4 | 4 | 5 | 6 | ✗ | 3<4<6 ✓ | 1 | 2 | 5 |

In [14]:
def trace_one_array(nums):
    n = len(nums)
    suf = nums[:]
    for i in range(n - 2, -1, -1):
        suf[i] = min(suf[i + 1], nums[i])
    print("suf =", suf)
    total, left_max = 0, nums[0]
    for i in range(1, n - 1):
        x = nums[i]
        c2 = left_max < x < suf[i + 1]
        c1 = nums[i - 1] < x < nums[i + 1]
        add = 2 if c2 else (1 if c1 else 0)
        total += add
        print(f"i={i} x={x} left_max={left_max} suf[i+1]={suf[i + 1]} đk2={c2} đk1={c1} -> +{add}, total={total}")
        left_max = max(left_max, x)
    return total

print("Đáp án:", trace_one_array([1, 5, 2, 3, 4, 6]))

suf = [1, 2, 2, 3, 4, 6]
i=1 x=5 left_max=1 suf[i+1]=2 đk2=False đk1=False -> +0, total=0
i=2 x=2 left_max=5 suf[i+1]=3 đk2=False đk1=False -> +0, total=0
i=3 x=3 left_max=5 suf[i+1]=4 đk2=False đk1=True -> +1, total=1
i=4 x=4 left_max=5 suf[i+1]=6 đk2=False đk1=True -> +1, total=2
Đáp án: 2


In [15]:
print("Đáp án:", trace_one_array([2, 4, 6, 4]))

suf = [2, 4, 4, 4]
i=1 x=4 left_max=2 suf[i+1]=4 đk2=False đk1=True -> +1, total=1
i=2 x=6 left_max=4 suf[i+1]=4 đk2=False đk1=False -> +0, total=1
Đáp án: 1


## 9. Code nộp LeetCode

In [16]:
class Solution:
    def sumOfBeauties(self, nums: List[int]) -> int:
        n = len(nums)
        suf = nums[:]                              # suf[i] = min(nums[i..n-1])
        for i in range(n - 2, -1, -1):
            suf[i] = min(suf[i + 1], nums[i])
        total, left_max = 0, nums[0]               # left_max = max(nums[0..i-1])
        for i in range(1, n - 1):
            x = nums[i]
            if left_max < x < suf[i + 1]:          # lớn hơn mọi bên trái, nhỏ hơn mọi bên phải
                total += 2
            elif nums[i - 1] < x < nums[i + 1]:    # chỉ so với hai láng giềng
                total += 1
            if x > left_max:
                left_max = x
        return total

for nums, exp in examples:
    assert Solution().sumOfBeauties(nums) == exp
print("Ví dụ đề: OK")

Ví dụ đề: OK


## 10. Vì sao thuật toán đúng

1. **Tương đương điều kiện 2.** "`nums[j] < nums[i]` với mọi `j < i`" ⇔ `max_{j<i} nums[j] < nums[i]` ⇔ `pre[i−1] < nums[i]` (vì max là một trong các phần tử, và mọi phần tử ≤ max). Tương tự "`nums[i] < nums[k]` với mọi `k > i`" ⇔ `nums[i] < suf[i+1]`.
2. **Bất biến `left_max`.** Trước vòng `i = 1`, `left_max = nums[0] = pre[0]`. Cuối vòng `i`, `left_max = max(pre[i−1], nums[i]) = pre[i]`. Vậy ở đầu vòng `i`, `left_max = pre[i−1]`.
3. **Mảng `suf` đúng** theo truy hồi `suf[i] = min(nums[i], suf[i+1])`, quy nạp từ `i = n−1`.
4. **Điểm số.** `if/elif` cộng đúng `2` khi điều kiện 2 thỏa, đúng `1` khi chỉ điều kiện láng giềng thỏa, và `0` còn lại — đúng định nghĩa. Tổng trên `i = 1..n−2` là đáp án.

## 11. Phân tích Time / Space

Gọi `n = len(nums)`.

| Cách | Time | Space phụ |
|---|---|---|
| 1. Brute force | `O(n²)` worst (mảng tăng dần) | `O(1)` |
| 2. `pre` + `suf` | `O(n)` | `O(n)` (2 mảng) |
| 3. `suf` + `left_max` | `O(n)` | `O(n)` (1 mảng) |
| 4. Hai mảng cờ | `O(n)` | `O(n)` |

Brute force có thể nhanh trên dữ liệu ngẫu nhiên nhờ `all` dừng sớm, nhưng worst case (mảng tăng dần) là bậc hai.

In [17]:
rng = random.Random(2012)
big_inc = list(range(1, 10**5 + 1))
big_rand = [rng.randint(1, 10**5) for _ in range(10**5)]
for name, fn in [("two_arrays", beauty_two_arrays), ("one_array", beauty_one_array), ("flags", beauty_flags),
                 ("Solution", lambda a: Solution().sumOfBeauties(a))]:
    t = time.perf_counter(); r1 = fn(big_inc); r2 = fn(big_rand)
    print(f"{name:10s}: tăng dần={r1}, ngẫu nhiên={r2} ({time.perf_counter() - t:.3f}s)")

two_arrays: tăng dần=199996, ngẫu nhiên=16624 (0.043s)
one_array : tăng dần=199996, ngẫu nhiên=16624 (0.031s)
flags     : tăng dần=199996, ngẫu nhiên=16624 (0.029s)
Solution  : tăng dần=199996, ngẫu nhiên=16624 (0.028s)


## 12. Bộ test

Ví dụ đề; biên (`n = 3`, mảng hằng, tăng dần, giảm dần, có phần tử bằng nhau ngay cạnh); đối chiếu brute force trên mảng ngẫu nhiên (giá trị nhỏ để có nhiều lặp; seed cố định).

In [18]:
fns = {
    "brute": beauty_brute, "two_arrays": beauty_two_arrays, "one_array": beauty_one_array, "flags": beauty_flags,
    "Solution": lambda a: Solution().sumOfBeauties(a),
    "SolutionTwoArrays": lambda a: SolutionTwoArrays().sumOfBeauties(a),
    "SolutionOneArray": lambda a: SolutionOneArray().sumOfBeauties(a),
}
edge = [([1, 1, 1], 0), ([1, 2, 2], 0), ([1, 2, 3, 4, 5], 6), ([5, 4, 3, 2, 1], 0),
        ([1, 3, 2, 4], 0), ([1, 2, 3, 2, 4, 5], 3), ([100000, 1, 100000], 0)]
for nums, exp in examples + edge:
    for name, fn in fns.items():
        assert fn(nums) == exp, (name, nums, fn(nums), exp)
print("Ví dụ + biên: OK")

Ví dụ + biên: OK


In [19]:
rng = random.Random(42)
for _ in range(1000):
    n = rng.randint(3, 25)
    hi = rng.choice([3, 5, 10, 100000])
    nums = [rng.randint(1, hi) for _ in range(n)]
    if rng.random() < 0.3:
        nums.sort()                               # mảng (gần) tăng để có nhiều điểm 2
    want = beauty_brute(nums)
    for name, fn in fns.items():
        assert fn(nums) == want, (name, nums)
print("1000 test ngẫu nhiên: mọi cách khớp brute force")

1000 test ngẫu nhiên: mọi cách khớp brute force


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Dùng `≤` thay cho `<`

Đề yêu cầu **nghiêm ngặt**. Với `[1, 2, 2]`, `nums[1] = 2` không nhỏ hơn `nums[2] = 2`, độ đẹp phải là `0`.

In [20]:
def wrong_le(nums):
    n = len(nums); suf = nums[:]
    for i in range(n - 2, -1, -1): suf[i] = min(suf[i + 1], nums[i])
    total, lm = 0, nums[0]
    for i in range(1, n - 1):
        if lm <= nums[i] <= suf[i + 1]: total += 2          # SAI: <=
        elif nums[i - 1] <= nums[i] <= nums[i + 1]: total += 1
        lm = max(lm, nums[i])
    return total

for a in ([1, 2, 2], [1, 1, 1]):
    print(a, "sai:", wrong_le(a), "| đúng:", Solution().sumOfBeauties(a))

[1, 2, 2] sai: 2 | đúng: 0
[1, 1, 1] sai: 2 | đúng: 0


In [21]:
print("sửa:", [Solution().sumOfBeauties(a) for a in ([1, 2, 2], [1, 1, 1])])

sửa: [0, 0]


### Lỗi 2 — Lấy max tiền tố **bao gồm cả** `nums[i]`

Nếu dùng `pre[i]` (đã gồm `nums[i]`) thay cho `pre[i−1]`, điều kiện `pre[i] < nums[i]` **không bao giờ** đúng ⇒ mất hết điểm 2.

In [22]:
def wrong_include_self(nums):
    n = len(nums)
    pre = list(accumulate(nums, max)); suf = list(accumulate(reversed(nums), min))[::-1]
    total = 0
    for i in range(1, n - 1):
        if pre[i] < nums[i] < suf[i]:             # SAI: pre[i], suf[i] gồm cả nums[i]
            total += 2
        elif nums[i - 1] < nums[i] < nums[i + 1]:
            total += 1
    return total

print("sai:", wrong_include_self([1, 2, 3]), "| đúng:", Solution().sumOfBeauties([1, 2, 3]))

sai: 1 | đúng: 2


In [23]:
print("sửa (pre[i-1], suf[i+1]):", beauty_two_arrays([1, 2, 3]))

sửa (pre[i-1], suf[i+1]): 2


### Lỗi 3 — Cộng cả 2 và 1 (dùng hai `if` độc lập)

Điều kiện 2 kéo theo điều kiện láng giềng, nên hai `if` riêng sẽ cộng `3` cho các phần tử đẹp nhất.

In [24]:
def wrong_two_ifs(nums):
    n = len(nums); suf = nums[:]
    for i in range(n - 2, -1, -1): suf[i] = min(suf[i + 1], nums[i])
    total, lm = 0, nums[0]
    for i in range(1, n - 1):
        if lm < nums[i] < suf[i + 1]: total += 2
        if nums[i - 1] < nums[i] < nums[i + 1]: total += 1   # SAI: phải là elif
        lm = max(lm, nums[i])
    return total

print("sai:", wrong_two_ifs([1, 2, 3]), "| đúng:", Solution().sumOfBeauties([1, 2, 3]))

sai: 3 | đúng: 2


In [25]:
print("sửa (elif):", Solution().sumOfBeauties([1, 2, 3]))

sửa (elif): 2


### Lỗi 4 — Duyệt cả hai đầu mút `i = 0` và `i = n−1`

Với `i = 0`, đoạn bên trái **rỗng** nên `all([])` là `True` ("đúng một cách trống rỗng") ⇒ phần tử đầu được cộng oan `2` điểm nếu nó nhỏ hơn mọi phần tử sau. Hơn nữa Python cho phép `nums[-1]` (phần tử cuối) nên `nums[i−1]` ở `i = 0` **không báo lỗi** mà âm thầm so với cuối mảng; còn `i = n−1` thì `nums[i+1]` báo `IndexError`. Bản sai dưới đây chỉ thêm `i = 0`.

In [26]:
def wrong_range(nums):
    n = len(nums); total = 0
    for i in range(0, n - 1):                     # SAI: bắt đầu từ 0
        if all(nums[j] < nums[i] for j in range(i)) and all(nums[i] < nums[k] for k in range(i + 1, n)):
            total += 2
        elif nums[i - 1] < nums[i] < nums[i + 1]:  # i = 0 -> nums[-1] là phần tử CUỐI
            total += 1
    return total

print("sai:", wrong_range([1, 2, 3]), "| đúng:", Solution().sumOfBeauties([1, 2, 3]))

sai: 4 | đúng: 2


In [27]:
print("sửa (range(1, n-1)):", beauty_brute([1, 2, 3]))

sửa (range(1, n-1)): 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Lớn hơn mọi phần tử bên trái / nhỏ hơn mọi phần tử bên phải"** ⇒ **max tiền tố / min hậu tố**.
- Thông tin bên trái cập nhật được "trên đường đi" ⇒ một biến; thông tin bên phải cần **tính trước** ⇒ một mảng.
- Đọc kỹ: nghiêm ngặt hay không, `if/elif`, phạm vi chỉ số.
- Mẹo `float("inf")` làm lính canh cho `suf[n]` giúp khỏi xử lý biên.
- Pattern tương tự: "phần tử là pivot của Quicksort", "chia mảng sao cho max(trái) ≤ min(phải)".

## 15. Biến thể của bài

1. **LeetCode 915 — Partition Array into Disjoint Intervals:** tìm điểm cắt nhỏ nhất với `max(trái) ≤ min(phải)` — cùng hai mảng.
2. **LeetCode 769/768 — Max Chunks To Make Sorted:** đếm số điểm `i` có `max(nums[0..i]) ≤ min(nums[i+1..])`.
3. **Đếm số phần tử "pivot"** (lớn hơn mọi bên trái, nhỏ hơn mọi bên phải) — chính là đếm điểm 2.
4. **Láng giềng bán kính `k`** (bài tập 5): thay "hai láng giềng" bằng cửa sổ `k` phần tử mỗi bên ⇒ hàng đợi đơn điệu.

Minh họa biến thể 2 (Max Chunks To Make Sorted II, tự dựng — kiểm bằng brute force sắp xếp từng khối):

In [28]:
def max_chunks(arr):
    n = len(arr); suf = arr[:] + [float("inf")]
    for i in range(n - 1, -1, -1): suf[i] = min(suf[i + 1], arr[i])
    cnt, lm = 0, float("-inf")
    for i in range(n):
        lm = max(lm, arr[i])
        if lm <= suf[i + 1]: cnt += 1
    return cnt

def max_chunks_brute(arr):
    n = len(arr); target = sorted(arr); best = 0
    for mask in range(1 << (n - 1)):              # mask chọn các vị trí cắt
        cuts = [i + 1 for i in range(n - 1) if mask >> i & 1]
        parts, prev = [], 0
        for c in cuts + [n]:
            parts.append(sorted(arr[prev:c])); prev = c
        if sum(parts, []) == target: best = max(best, len(parts))
    return best

rng = random.Random(768)
for _ in range(300):
    arr = [rng.randint(1, 5) for _ in range(rng.randint(1, 9))]
    assert max_chunks(arr) == max_chunks_brute(arr)
print("Max Chunks II khớp brute force; [2,1,3,4,4] ->", max_chunks([2, 1, 3, 4, 4]))

Max Chunks II khớp brute force; [2,1,3,4,4] -> 4


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Brute force | `all(...)` hai phía | `O(n²)` | `O(1)` | kiểm tra |
| 2. `pre` + `suf` | hai mảng tiền/hậu tố | `O(n)` | `O(n)` | rõ ràng, dễ debug |
| 3. `suf` + `left_max` | một mảng + một biến | `O(n)` | `O(n)` (½) | **nộp bài** |
| 4. Cờ kỷ lục | `good_left`, `good_right` | `O(n)` | `O(n)` | khi cần đếm "kỷ lục hai phía" |

## 17. Bài tập (từ dễ đến khó)

1. **Đoán output:** `sumOfBeauties([1, 2, 3, 4, 5])`.
2. **Tính tay:** `sumOfBeauties([2, 1, 3, 5, 4, 6])` (lập bảng như mục 8).
3. **Điền TODO:** `beauty_list(nums)` trả về **danh sách** độ đẹp của từng `i = 1..n−2`.
4. **Tự viết lại:** viết lại Cách 3 không nhìn code.
5. **Mini challenge:** `beauty_k(nums, k)`: độ đẹp 2 như cũ; độ đẹp 1 khi `nums[i]` lớn hơn **`k` phần tử ngay trước** và nhỏ hơn **`k` phần tử ngay sau** (vẫn xét mọi `i = 1..n−2`; ở gần biên, cửa sổ bị cắt bớt cho vừa mảng). Yêu cầu `O(n)` bằng hàng đợi đơn điệu; kiểm bằng brute force.

In [29]:
# Bài 3 — TODO (chỉ định nghĩa, không gọi)
def beauty_list(nums):
    res = []
    # TODO
    return res

In [30]:
# Bài 4 — TODO (chỉ định nghĩa, không gọi)
def my_sum_of_beauties(nums):
    # TODO: suf + left_max
    pass

In [31]:
# Bài 5 — TODO (chỉ định nghĩa, không gọi)
def beauty_k(nums, k):
    # TODO: max của cửa sổ trái [i-k, i-1] và min của cửa sổ phải [i+1, i+k] bằng deque đơn điệu
    pass

## 18. Gợi ý 3 mức

**Bài 1.** (1) Mảng tăng nghiêm ngặt. (2) Mọi `i = 1..3` đều có điểm 2. (3) `3 · 2`.

**Bài 2.** (1) `suf = [1,1,3,4,4,6]`. (2) Chỉ `i = 2` (`x = 3`) có `max trái = 2 < 3 < 4`. (3) Các `i` khác không thỏa cả điều kiện láng giềng ⇒ tổng `2`.

**Bài 3.** (1) Dùng lại Cách 3. (2) `res.append(2 / 1 / 0)` thay cho cộng dồn. (3) `sum(res)` phải bằng đáp án.

**Bài 4.** (1) `suf` từ phải. (2) `left_max = nums[0]`. (3) Cập nhật `left_max` **sau** khi xét `i`.

**Bài 5.** (1) Cửa sổ trái của `i` là `[max(0, i−k), i−1]`, trượt sang phải mỗi bước. (2) Deque giảm dần giữ max; deque tăng dần cho min cửa sổ phải — duyệt từ phải sang. (3) Tiền tính `wmax_left[i]`, `wmin_right[i]` rồi áp dụng `if/elif`.

## 19. Đáp án tham khảo

In [32]:
# Bài 1, 2
assert Solution().sumOfBeauties([1, 2, 3, 4, 5]) == 6
assert Solution().sumOfBeauties([2, 1, 3, 5, 4, 6]) == 2
print("Bài 1: 6 | Bài 2: 2")
trace_one_array([2, 1, 3, 5, 4, 6])

Bài 1: 6 | Bài 2: 2
suf = [1, 1, 3, 4, 4, 6]
i=1 x=1 left_max=2 suf[i+1]=3 đk2=False đk1=False -> +0, total=0
i=2 x=3 left_max=2 suf[i+1]=4 đk2=True đk1=True -> +2, total=2
i=3 x=5 left_max=3 suf[i+1]=4 đk2=False đk1=False -> +0, total=2
i=4 x=4 left_max=5 suf[i+1]=6 đk2=False đk1=False -> +0, total=2


2

In [33]:
# Bài 3, 4
def beauty_list_ans(nums):
    n = len(nums); suf = nums[:]
    for i in range(n - 2, -1, -1): suf[i] = min(suf[i + 1], nums[i])
    res, lm = [], nums[0]
    for i in range(1, n - 1):
        if lm < nums[i] < suf[i + 1]: res.append(2)
        elif nums[i - 1] < nums[i] < nums[i + 1]: res.append(1)
        else: res.append(0)
        lm = max(lm, nums[i])
    return res

my_sum_of_beauties_ans = lambda nums: sum(beauty_list_ans(nums))
assert beauty_list_ans([2, 4, 6, 4]) == [1, 0]
rng = random.Random(3)
for _ in range(300):
    a = [rng.randint(1, 6) for _ in range(rng.randint(3, 15))]
    assert my_sum_of_beauties_ans(a) == beauty_brute(a)
print("Bài 3, 4: OK;", beauty_list_ans([1, 5, 2, 3, 4, 6]))

Bài 3, 4: OK; [0, 0, 1, 1]


In [34]:
# Bài 5
from collections import deque

def beauty_k_ans(nums, k):
    n = len(nums)
    wmax_left = [None] * n; dq = deque()          # deque chỉ số, giá trị giảm dần
    for i in range(n):
        while dq and dq[0] < i - k: dq.popleft()
        wmax_left[i] = nums[dq[0]] if dq else None
        while dq and nums[dq[-1]] <= nums[i]: dq.pop()
        dq.append(i)
    wmin_right = [None] * n; dq = deque()         # giá trị tăng dần, duyệt từ phải
    for i in range(n - 1, -1, -1):
        while dq and dq[0] > i + k: dq.popleft()
        wmin_right[i] = nums[dq[0]] if dq else None
        while dq and nums[dq[-1]] >= nums[i]: dq.pop()
        dq.append(i)
    suf = nums[:]
    for i in range(n - 2, -1, -1): suf[i] = min(suf[i + 1], nums[i])
    total, lm = 0, nums[0]
    for i in range(1, n - 1):
        if lm < nums[i] < suf[i + 1]: total += 2
        elif wmax_left[i] < nums[i] < wmin_right[i]: total += 1
        lm = max(lm, nums[i])
    return total

def beauty_k_brute(nums, k):
    n = len(nums); total = 0
    for i in range(1, n - 1):
        if max(nums[:i]) < nums[i] < min(nums[i + 1:]): total += 2
        elif max(nums[max(0, i - k):i]) < nums[i] < min(nums[i + 1:i + 1 + k]): total += 1
    return total

rng = random.Random(9)
for _ in range(500):
    a = [rng.randint(1, 8) for _ in range(rng.randint(3, 15))]; k = rng.randint(1, 4)
    assert beauty_k_ans(a, k) == beauty_k_brute(a, k)
    assert beauty_k_ans(a, 1) == Solution().sumOfBeauties(a)          # k = 1 trùng bài gốc
print("Bài 5: OK; ví dụ:", beauty_k_ans([1, 5, 2, 3, 4, 6], 2))

Bài 5: OK; ví dụ: 1


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template "so với mọi phần tử hai phía":**

```python
suf = nums[:]                               # min hậu tố (hoặc max, tùy đề)
for i in range(n - 2, -1, -1):
    suf[i] = min(suf[i + 1], nums[i])
left = nums[0]                              # max tiền tố chạy theo
for i in range(1, n - 1):
    if left < nums[i] < suf[i + 1]: ...
    elif nums[i - 1] < nums[i] < nums[i + 1]: ...
    left = max(left, nums[i])
```

In [35]:
def template(nums):
    n = len(nums); suf = nums[:]
    for i in range(n - 2, -1, -1): suf[i] = min(suf[i + 1], nums[i])
    left, total = nums[0], 0
    for i in range(1, n - 1):
        if left < nums[i] < suf[i + 1]: total += 2
        elif nums[i - 1] < nums[i] < nums[i + 1]: total += 1
        left = max(left, nums[i])
    return total
print([template(a) for a, _ in examples])

[2, 1, 0]


**Checklist:**

- [ ] So sánh **nghiêm ngặt** `<` ở cả hai điều kiện.
- [ ] Dùng `max(nums[0..i−1])` và `min(nums[i+1..n−1])` — **không** bao gồm `nums[i]`.
- [ ] `if ... elif ...` (không cộng 3).
- [ ] Chỉ duyệt `i = 1..n−2`.
- [ ] Cập nhật `left_max` **sau** khi xử lý `i`.
- [ ] Đã thử mảng hằng, tăng dần, giảm dần, `n = 3`.